[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/moeabench/blob/main/examples/example_full.ipynb)

# API Walkthrough

This notebook mirrors [`examples/example_full.py`](./example_full.py): two experiments, the canonical stats API, and a full tour through topology, performance, attainment, strata, and clinic diagnostics.


In [ ]:
# Install moeabench from GitHub
!pip install --quiet git+https://github.com/monacofj/moeabench.git


In [ ]:
from pathlib import Path

import moeabench as mb


In [ ]:
mb.system.version()


## Setup: same MOP, two MOEAs

We prepare two experiments over the same benchmark so that every later comparison isolates algorithmic behavior rather than problem differences.


In [ ]:
mop = mb.mops.DTLZ2(M=3)
#mop.calibrate()  # Ensure baseline/GT is available for this M (e.g., M=4).
exp1 = mb.experiment()
exp1.name = "NSGA-II"
exp1.mop = mop
exp1.moea = mb.moeas.NSGA2(population=100, generations=80)


In [ ]:
exp2 = mb.experiment()
exp2.name = "NSGA-III"
exp2.mop = mop
exp2.moea = mb.moeas.NSGA3(population=100, generations=80)


In [ ]:
exp1_zip = Path("example_full_exp1.zip")
exp2_zip = Path("example_full_exp2.zip")


In [ ]:
if exp1_zip.exists():
    exp1.load(str(exp1_zip))
else:
    exp1.run(repeat=5)
    exp1.save(str(exp1_zip))


In [ ]:
if exp2_zip.exists():
    exp2.load(str(exp2_zip))
else:
    exp2.run(repeat=5)
    exp2.save(str(exp2_zip))


## 0. Topology

Start by inspecting both experiments together in objective space. This gives the fastest visual read of convergence, spread, and agreement with the inferred Ground Truth.


In [ ]:
mb.view.topology(exp1, exp2)                # Front clouds + inferred GT.;
# Observe overlap with GT (convergence) and cloud spread (diversity).


## 1. Metrics, reports, and plots

For each metric, we follow the same pattern: compute the metric objects, inspect their narrative reports, and then visualize convergence, spread, and density through the canonical views.


### Hypervolume

Hypervolume summarizes dominated volume and is often the most compact scalar picture of overall convergence plus diversity.


In [ ]:
hv1 = mb.metrics.hv(exp1, scale="abs")  # Falls back to raw if absolute is unavailable.
hv2 = mb.metrics.hv(exp2, scale="abs")  # Falls back to raw if absolute is unavailable.
hv1.report()                                 # exp1 HV summary.;
hv2.report()                                 # exp2 HV summary.;
mb.view.history(hv1, hv2)                    # Convergence.;


In [ ]:
hv_shift = mb.stats.perf_shift(hv1, hv2)     # MW shift test.
hv_shift.report()                            # Shift evidence.;
mb.view.spread(hv_shift)                     # Final spread + shift.;


In [ ]:
hv_match = mb.stats.perf_match(hv1, hv2)     # KS match test.
hv_match.report()                            # Match verdict.;
mb.view.density(hv_match)                    # Density shape + KS verdict.;


In [ ]:
hv_win = mb.stats.perf_win(hv1, hv2)         # A12 win prob.
hv_win.report()                              # Effect size.;


### GD

Generational Distance highlights how far the obtained front remains from the reference front.


In [ ]:
gd1 = mb.metrics.gd(exp1)                    # GD trajectory for exp1.
gd2 = mb.metrics.gd(exp2)                    # GD trajectory for exp2.
gd1.report()                                 # exp1 GD summary.;
gd2.report()                                 # exp2 GD summary.;
mb.view.history(gd1, gd2)                    # Convergence.;


In [ ]:
gd_shift = mb.stats.perf_shift(gd1, gd2)     # MW shift test.
gd_shift.report()                            # Shift evidence.;
mb.view.spread(gd_shift)                    # Final spread + shift.;


In [ ]:
gd_match = mb.stats.perf_match(gd1, gd2)     # KS match test.
gd_match.report()                            # Match verdict.;
mb.view.density(gd_match)                    # Density shape + KS verdict.;


In [ ]:
gd_win = mb.stats.perf_win(gd1, gd2)         # A12 win prob.
gd_win.report()                              # Effect size.;


### GD+

GD+ refines GD by respecting Pareto-compliant distance behavior in dominated regions.


In [ ]:
gdplus1 = mb.metrics.gdplus(exp1)            # GD+ trajectory for exp1.
gdplus2 = mb.metrics.gdplus(exp2)            # GD+ trajectory for exp2.
gdplus1.report()                             # exp1 GD+ summary.;
gdplus2.report()                             # exp2 GD+ summary.;
mb.view.history(gdplus1, gdplus2)            # Convergence.;


In [ ]:
gdplus_shift = mb.stats.perf_shift(gdplus1, gdplus2)    # MW shift test.
gdplus_shift.report()                        # Shift evidence.;
mb.view.spread(gdplus_shift)                 # Final spread + shift.;


In [ ]:
gdplus_match = mb.stats.perf_match(gdplus1, gdplus2)    # KS match test.
gdplus_match.report()                        # Match verdict.;
mb.view.density(gdplus_match)                # Density shape + KS verdict.;


In [ ]:
gdplus_win = mb.stats.perf_win(gdplus1, gdplus2)        # A12 win prob.
gdplus_win.report()                          # Effect size.;


### IGD

Inverted Generational Distance asks how well the solver covers the reference front itself.


In [ ]:
igd1 = mb.metrics.igd(exp1)                  # IGD trajectory for exp1.
igd2 = mb.metrics.igd(exp2)                  # IGD trajectory for exp2.
igd1.report()                                # exp1 IGD summary.;
igd2.report()                                # exp2 IGD summary.;
mb.view.history(igd1, igd2)                  # Convergence.;


In [ ]:
igd_shift = mb.stats.perf_shift(igd1, igd2)  # MW shift test.
igd_shift.report()                           # Shift evidence.;
mb.view.spread(igd_shift)                    # Final spread + shift.;


In [ ]:
igd_match = mb.stats.perf_match(igd1, igd2)  # KS match test.
igd_match.report()                           # Match verdict.;
mb.view.density(igd_match)                   # Density shape + KS verdict.;


In [ ]:
igd_win = mb.stats.perf_win(igd1, igd2)      # A12 win prob.
igd_win.report()                             # Effect size.;


### IGD+

IGD+ is the Pareto-compliant counterpart of IGD for more robust front-to-reference comparisons.


In [ ]:
igdplus1 = mb.metrics.igdplus(exp1)          # IGD+ trajectory for exp1.
igdplus2 = mb.metrics.igdplus(exp2)          # IGD+ trajectory for exp2.
igdplus1.report()                            # exp1 IGD+ summary.;
igdplus2.report()                            # exp2 IGD+ summary.;
mb.view.history(igdplus1, igdplus2)          # Convergence.;


In [ ]:
igdplus_shift = mb.stats.perf_shift(igdplus1, igdplus2)    # MW shift test.
igdplus_shift.report()                       # Shift evidence.;
mb.view.spread(igdplus_shift)                # Final spread + shift.;


In [ ]:
igdplus_match = mb.stats.perf_match(igdplus1, igdplus2)    # KS match test.
igdplus_match.report()                       # Match verdict.;
mb.view.density(igdplus_match)               # Density shape + KS verdict.;


In [ ]:
igdplus_win = mb.stats.perf_win(igdplus1, igdplus2)        # A12 win prob.
igdplus_win.report()                         # Effect size.;


### EMD

Earth Mover's Distance captures geometric displacement between distributions rather than only nearest-point proximity.


In [ ]:
emd1 = mb.metrics.emd(exp1)                  # EMD trajectory for exp1.
emd2 = mb.metrics.emd(exp2)                  # EMD trajectory for exp2.
emd1.report()                                # exp1 EMD summary.;
emd2.report()                                # exp2 EMD summary.;
mb.view.history(emd1, emd2)                  # Convergence.;


In [ ]:
emd_shift = mb.stats.perf_shift(emd1, emd2)  # MW shift test.
emd_shift.report()                           # Shift evidence.;
mb.view.spread(emd_shift)                    # Final spread + shift.;


In [ ]:
emd_match = mb.stats.perf_match(emd1, emd2)  # KS match test.
emd_match.report()                           # Match verdict.;
mb.view.density(emd_match)                   # Density shape + KS verdict.;


In [ ]:
emd_win = mb.stats.perf_win(emd1, emd2)      # A12 win prob.
emd_win.report()                             # Effect size.;


### Front size

Front ratio summarizes how much of the population remains non-dominated over time.


In [ ]:
fsize1 = mb.metrics.front_ratio(exp1)        # Front-size trajectory for exp1.
fsize2 = mb.metrics.front_ratio(exp2)        # Front-size trajectory for exp2.
fsize1.report()                              # exp1 front-size summary.;
fsize2.report()                              # exp2 front-size summary.;
mb.view.history(fsize1, fsize2)              # Convergence.;


In [ ]:
fsize_shift = mb.stats.perf_shift(fsize1, fsize2)  # MW shift test.
fsize_shift.report()                               # Shift evidence.;
mb.view.spread(fsize_shift)                  # Final spread + shift.;


In [ ]:
fsize_match = mb.stats.perf_match(fsize1, fsize2)  # KS match test.
fsize_match.report()                               # Match verdict.;
mb.view.density(fsize_match)                 # Density shape + KS verdict.;


In [ ]:
fsize_win = mb.stats.perf_win(fsize1, fsize2)      # A12 win prob.
fsize_win.report()                                 # Effect size.;
# Observe trajectory speed (history), final contrast (spread), and tails (density).


## 2. Topology compare aliases

These aliases expose three complementary questions about geometric agreement: match, tail sensitivity, and spatial displacement.


### KS-style topological match


In [ ]:
topo_match = mb.stats.topo_match(exp1, exp2) # KS equivalence test.
topo_match.report()                          # Objective-space match verdict.;


### Anderson-style tail check


In [ ]:
topo_tail = mb.stats.topo_tail(exp1, exp2)   # Anderson tail test.
topo_tail.report()                           # Tail-sensitive match verdict.;


### EMD-style spatial shift


In [ ]:
topo_shift = mb.stats.topo_shift(exp1, exp2, threshold=0.05)  # EMD displacement test.
topo_shift.report()                                            # Spatial shift diagnosis.;


In [ ]:
topo_match_obj = mb.stats.topo_match(exp1, exp2, axes=[0])    # Single-objective axis test.
topo_match_var = mb.stats.topo_match(exp1, exp2, space="vars", axes=[0])  # Decision-axis test.
mb.view.density(topo_match_obj)              # Objective-axis density.;
mb.view.density(topo_match_var)              # Decision-axis density.;
# Observe objective-space equivalence versus possible decision-space divergence.


## 3. Attainment and gap

Attainment analysis summarizes stochastic reachability, while the gap view localizes where one experiment dominates the other in space.


In [ ]:
att1 = mb.stats.attainment(exp1)                    # Median attainment surface for exp1.
att2 = mb.stats.attainment(exp2)                    # Median attainment surface for exp2.
band1_lo = mb.stats.attainment(exp1, level=0.1)     # Lower envelope for exp1.
band1_hi = mb.stats.attainment(exp1, level=0.9)     # Upper envelope for exp1.
band2_lo = mb.stats.attainment(exp2, level=0.1)     # Lower envelope for exp2.
band2_hi = mb.stats.attainment(exp2, level=0.9)     # Upper envelope for exp2.
gap = mb.stats.attainment_gap(exp1, exp2)           # Localized attainment difference.
gap.report()                                        # Gap diagnosis.;


In [ ]:
mb.view.bands(att1, band1_lo, band1_hi, att2, band2_lo, band2_hi, style="fill")  # Reliability corridor.;
mb.view.topology(att1, att2)                 # Median surfaces in objective space.;
mb.view.gap(gap)                             # Signed local superiority map.;
# Observe corridor width (reliability) and localized superiority regions (gap).


## 4. Strata

Structural analysis reorganizes both experiments into shared dominance layers, making internal population quality visible instead of only final-front geometry.


In [ ]:
ranks = mb.stats.ranks(exp1, exp2)           # Rank depth and pressure.
ranks.report()                               # Rank structure summary.;
mb.view.ranks(ranks)                         # Rank occupancy bars.;


In [ ]:
strata = mb.stats.strata(exp1, exp2)         # Rank-wise quality distribution.
strata.report()                              # Strata distribution summary.;
mb.view.strata(strata)                       # Rank quality box summaries.;


In [ ]:
tiers = mb.stats.tiers(exp1, exp2)           # Shared-tier duel between both groups.
tiers.report()                               # Tier duel summary.;
mb.view.tiers(tiers)                         # Shared-tier stacked duel.;
# Observe selection pressure depth and class occupancy profile.


## 5. Clinic audit

The clinical layer turns the raw analytical objects into a synthetic diagnostic narrative, combining FAIR metrics, Q-scores, and pathology-oriented views.


In [ ]:
diag1 = mb.clinic.audit(exp1)                # Clinical synthesis for exp1.
diag2 = mb.clinic.audit(exp2)                # Clinical synthesis for exp2.
diag1.report(full=True)                      # Full audit narrative.;
diag2.report(full=True)                      # Full audit narrative.;


In [ ]:
mb.view.radar(diag1, diag2)                  # Global health radar.;


In [ ]:
close1 = mb.clinic.closeness(exp1)           # Closeness pathology for exp1.
close2 = mb.clinic.closeness(exp2)           # Closeness pathology for exp2.
close1.report()                              # Closeness summary.;
close2.report()                              # Closeness summary.;



In [ ]:
mb.view.ecdf(close1)                         # Quantile profile.;
mb.view.ecdf(close2)                         # Quantile profile.;


In [ ]:
mb.view.density(close1)                      # Distribution morphology.;
mb.view.density(close2)                      # Distribution morphology.;


In [ ]:
mb.view.history(close1)                      # Temporal pathology evolution.;
mb.view.history(close2)                      # Temporal pathology evolution.;
